# Chapter 6 experiment harness

Reproduces the experiment results chapter 6 reports: tables 6.1, 6.4 and 6.6 to 6.12,
figures 6.3, 6.4 and 6.6, and the exactness counts quoted beside table 6.4. Tables
6.2, 6.3 and 6.5 hold definitions and settings, and figures 6.1, 6.2 and 6.5 are
diagrams, so nothing here generates them.

Every convolution runs on the GPU through the chapter's own listings, and every VHC
decision is made by `receive_halo`, `advance`, `ready_strips`, `check_for_missing`
and `accept_final`. Time is simulated: worker compute times come from the kernel
costs measured on the book's RTX 5070 (`model/costs.json`), and delivery times,
worker speeds and failures are seeded draws.

## Environment

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

import conv_experiments as ex

print("GPU                 :", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
print("evidence class      : simulated_decentralized")
print("  the convolutions are real; delivery times, speeds and failures are seeded draws")

plt.rcParams.update({"font.size": 12, "axes.labelsize": 13, "figure.dpi": 110})
INK, MUTED, EXACT, PAD, MISSING = "#1f2933", "#8a94a6", "#2c8a4b", "#d9dee5", "#c0392b"


def table(title, header, rows):
    print(title)
    print("  " + "".join(f"{h:>{w}}" for h, w in header))
    for r in rows:
        print("  " + "".join(f"{v:>{w}}" for v, (_h, w) in zip(r, header)))


def fmt(x, places=2, scale=1):
    """Round half up on the exact decimal value, as the book's tables do."""
    d = Decimal(repr(float(x))) * scale
    return str(d.quantize(Decimal(1).scaleb(-places), rounding=ROUND_HALF_UP))


def pct(x, places=1):
    return fmt(x, places, 100) + "%"

## Table 6.1 Kernel time on one 4,096 x 4,096 image

Listings 6.1, 6.3 and 6.4 and cuDNN (through PyTorch, TensorFloat-32 off), median of
200 launches after a warm-up. Times depend on the GPU and vary from run to run, so
they will not match the book's RTX 5070 exactly.

In [ ]:
k = ex.kernel_ladder()
names = ("basic", "constant", "tiled_16", "tiled_32", "cudnn")
table(f"Table 6.1 Median kernel time (ms), {k['device']}",
      (("filter side", 12), ("direct", 9), ("constant", 10), ("tiled 16", 10),
       ("tiled 32", 10), ("cuDNN", 9)),
      [(r["filter_side"], *(fmt(r[n]["median_ms"], 3) for n in names)) for r in k["rows"]])
print("\nlargest difference from a float64 reference:",
      f"{max(r[n]['max_abs_error'] for r in k['rows'] for n in names):.1e}")
print("listings 6.2 and 6.5 agree with the timed launches:", k["listings_agree"])

## Table 6.4 Listing 6.6 against cuDNN on each layer

One image, every layer of the network, same timing method.

In [ ]:
lk = ex.layer_kernels()
table("Table 6.4 Listing 6.6 against cuDNN (ms)",
      (("layer", 6), ("channels", 11), ("side", 6), ("multiply-adds", 15),
       ("listing 6.6", 13), ("cuDNN", 9)),
      [(r["layer"] + 1, f"{r['c_in']}, {r['c_out']}", r["side"], f"{r['macs']:,}",
        fmt(r["ours_median_ms"], 3), fmt(r["cudnn_median_ms"], 3)) for r in lk])
print("\nlargest difference from cuDNN:", f"{max(r['max_abs_difference'] for r in lk):.1e}")

## Figure 6.3 A photograph through two edge filters and the trained network

The photograph (Stefan van der Walt, CC0, distributed with scikit-image) through
listing 6.5 with the two Sobel filters, and through the trained network one layer at a
time with listing 6.7. Each map is scaled to its own range.

In [ ]:
m = ex.feature_maps()
fig, axes = plt.subplots(3, 3, figsize=(10.5, 7.6))
panels = [(np.abs(m["edges"]["vertical"]), "|Sobel x|: vertical edges"),
          (np.abs(m["edges"]["horizontal"]), "|Sobel y|: horizontal edges")]
panels += [(m["layers"][0][c], f"convolution 1, channel {c}") for c in (14, 26, 29)]
panels += [(m["layers"][5][c], f"convolution 6, channel {c}") for c in (18, 6, 17)]
axes[0, 0].imshow(m["rgb"])
axes[0, 0].add_patch(Rectangle((150, 112), 32, 32, fill=False, ec="#f2c94c", lw=1.8))
axes[0, 0].set_title("input, 448 x 300", fontsize=10.5)
for ax, (a, title) in zip(axes.flat[1:], panels):
    ax.imshow(a, cmap="gray", vmin=0, vmax=max(float(np.percentile(a, 99.5)), 1e-6))
    ax.set_title(title, fontsize=10.5)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
plt.show()

## Exactness over the validation set

Listing 6.8 on one GPU, listing 6.10 across 16 workers, and VHC under each halo
schedule, compared bitwise on every validation image, and listing 6.8's class
compared with PyTorch and cuDNN. All 3,925 images take over an hour; set `LIMIT` to a
smaller number for a quick check.

In [ ]:
LIMIT = None                       # None: all 3,925 validation images
e = ex.exactness(limit=LIMIT)
n = e["images"]
print(f"images                                   : {n:,}")
print(f"listing 6.10 bitwise equal to listing 6.8: {e['distributed_bitwise']:,}")
for s, c in e["vhc_bitwise"].items():
    print(f"VHC, exchange {s:<12s} bitwise equal   : {c:,}")
print(f"same class as PyTorch/cuDNN              : {e['top1_equal_cudnn']:,}")
print(f"largest logit difference from cuDNN      : {e['max_logit_difference_cudnn']:.1e}")
print(f"top-1 accuracy, listing 6.8              : {pct(e['correct_ours'] / n)}")

## Table 6.6 and figure 6.4

Six conditions, 20 requests each, the conventional exchange and VHC on the same
draws. A dash means the arm did not complete within the five-second window.

In [ ]:
w = ex.working_example()
table("Table 6.6 Conventional exchange and VHC with missing and late workers",
      (("condition", 30), ("conventional T100 (s)", 23), ("VHC T100 (s)", 14),
       ("repairs", 9), ("exact before repair", 21), ("bound", 8)),
      [(r["condition"], fmt(r["conventional_t100"]) if r["conventional_completed"] else "-",
        fmt(r["vhc_t100"]), fmt(r["vhc_repairs"]),
        pct(r["vhc_area_before_repair"]) if r["vhc_area_before_repair"] is not None else "-",
        pct(r["independent_bound"])) for r in w["rows"]])
print("\nevery VHC output bitwise equal to listing 6.8:", all(r["bitwise"] for r in w["rows"]))

In [ ]:
t = np.array(w["t"])
fig, ax = plt.subplots(figsize=(8.6, 4.8))
colours = {"One interior tile missing": "#2f6db5", "One corner tile missing": "#2c8a4b",
           "Two adjacent tiles missing": "#7a4fa3"}
ax.plot(t, w["curves"]["All tiles present"]["vhc"], color=INK, lw=1.6, label="all tiles present, VHC")
ax.plot(t, w["curves"]["All tiles present"]["conventional"], color=MUTED, lw=1.6, ls="--",
        label="all tiles present, conventional")
for name, col in colours.items():
    ax.plot(t, w["curves"][name]["vhc"], color=col, lw=1.8, label=f"{name.lower()}, VHC")
ax.plot(t, w["curves"]["One interior tile missing"]["conventional"], color=MISSING, lw=1.8,
        label="any tile missing, conventional")
ax.set_xlabel("seconds")
ax.set_ylabel("exact fraction of the final feature map")
ax.set_ylim(-0.03, 1.05)
ax.legend(frameon=False, loc="lower right", fontsize=10)
ax.set_title("Figure 6.4 Exact area of the final feature map over time", fontsize=12)
plt.show()

## Figure 6.6 Which final cells are exact with tile 5 missing

One request, tile 5 missing and no spare: the cells independent of tile 5, the cells
VHC makes exact sending cells as they become exact, and the cells it makes exact when
each band is sent only once complete.

In [ ]:
m = ex.independence_maps()
fig, axes = plt.subplots(1, 3, figsize=(11, 4.0))
for ax, key, title in zip(axes, ("bound", "cells", "strip"),
                          ("independent of tile 5", "exact, cells", "exact, whole bands")):
    a = np.array(m[key])
    ax.imshow(a, cmap=ListedColormap([PAD, EXACT]), vmin=0, vmax=1)
    ax.add_patch(Rectangle((7.5, 7.5), 8, 8, fill=False, ec=MISSING, lw=2))
    ax.set_title(f"{title}: {a.mean():.1%}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Table 6.7 Injected halo commits

In [ ]:
f = ex.fault_injection()
table("Table 6.7 Injected halo commits and how listing 6.14 classifies them",
      (("injected commit", 46), ("classified", 12)),
      [(r["injection"], r["status"]) for r in f["rows"]])
print("\nall as expected:", f["all_as_expected"], "| output bitwise equal:", f["output_bitwise"])

## Table 6.8 Halo depth, every tile present

In [ ]:
h = ex.halo_depth()
table("Table 6.8 Halo depth against messages, bytes, redundant computation and T100",
      (("link", 14), ("exchange", 13), ("messages", 10), ("halo bytes", 12),
       ("redundant", 11), ("T100 (ms)", 11)),
      [(r["link"], r["schedule"], f"{round(r['messages']):,}", f"{round(r['halo_bytes']):,}",
        fmt(r["redundant_compute"]), fmt(r["t100"] * 1000, 2)) for r in h["rows"]])

## Table 6.9 Whole bands against exact cells

In [ ]:
g = ex.granularity()
table("Table 6.9 Whole bands against exact cells",
      (("sending", 13), ("messages", 10), ("halo bytes", 12), ("T100 (s)", 10),
       ("exact, tile 5 missing", 23)),
      [("whole bands" if r["granularity"] == "strip" else "exact cells", f"{round(r['messages']):,}",
        f"{round(r['halo_bytes']):,}", fmt(r["t100"], 3), pct(r["exact_area_one_missing"]))
       for r in g["rows"]])
print("\nindependence bound:", pct(g["independent_bound"]))

## Table 6.10 Repair deadline, tile 5 stalled for 1 s

In [ ]:
rd = ex.repair_deadline()
table("Table 6.10 Repair deadline against T100 and repairs",
      (("repair deadline (s)", 21), ("T100 (s)", 10), ("repairs", 9)),
      [(fmt(r["repair_deadline"]), fmt(r["t100"]), fmt(r["repairs"])) for r in rd["rows"]])

## Table 6.11 Substituted halos from tile 5

The panning stream: frame f is the crop starting f pixels from the left. Tile 5's
halos are replaced by its halos from frame f - s, or by zeros. 200 images, frames 8,
16, 24 and 32. It takes about an hour on an RTX 5070.

In [ ]:
st = ex.staleness()
label = {"padding": "zeros (padding)"}
table("Table 6.11 Substituted halos from tile 5 against the exact result",
      (("substitution", 18), ("same class", 12), ("relative logit error", 22)),
      [(label.get(r["mode"], "frame f - " + r["mode"].split()[1]), pct(r["top1_agreement"]),
        fmt(r["mean_relative_logit_error"], 4)) for r in st["rows"]])

## Table 6.12 Reach of one altered halo value

In [ ]:
c = ex.containment()
table("Table 6.12 Cells changed by one altered halo value, against the predicted reach",
      (("convolution", 13), ("side", 6), ("predicted", 11), ("changed, mean", 15),
       ("changed, max", 14)),
      [(r["layer"] + 1, r["side"], f"{r['predicted_cells']:,}", fmt(r["mean_differing_cells"], 1),
        r["max_differing_cells"]) for r in c["rows"]])
print("\nchanged cells outside the predicted reach:", c["cells_outside_cone"])

## What the harness does not show

**The network.** Workers are seeded delivery times and speeds, not machines, and a
spare's start-up is a fixed 0.45 seconds. Nothing crosses a network. That is what
`simulated_decentralized` means.

**Wall-clock coordination.** `vhc_infer.py` runs listing 6.19 for real, one thread per
tile, with tile 5's worker lost and a spare taking its slot: `python vhc_infer.py`.
Listing 6.10 runs from `distributed_cnn.py`, and listing 6.11 under MPI:
`mpiexec -n 16 python collective_cnn.py`.

**Trust.** The checks in listing 6.14 establish that a commit is well formed and
belongs where it claims, not that its values are the ones its tile computed.